In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [2]:
# Reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device("cpu")

# Helper function
def accuracy_from_probs(probs, targets, threshold=0.5):
    preds = (probs >= threshold).float()
    return (preds == targets).float().mean().item()

# 1. Perceptron for Logic Gate Classification

In [15]:
# Inputs
X = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)

# Outputs
Y_and = torch.tensor([[0],[0],[0],[1]], dtype=torch.float32)
Y_or  = torch.tensor([[0],[1],[1],[1]], dtype=torch.float32)
Y_xor = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

class Perceptron(nn.Module):
    def __init__(self):
        super(Perceptron, self).__init__()
        self.fc = nn.Linear(2, 1)   # 2 inputs → 1 output
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        return self.sigmoid(self.fc(x))

def train_model(X, Y, epochs=1000, lr=0.1):
    model = Perceptron()
    criterion = nn.BCELoss()
    optimizer = optim.Adam(params=model.parameters(), lr=lr)
    
    for _ in range(epochs):
        # Forward pass
        outputs = model(X)
        loss = criterion(outputs, Y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    return model

model_and = train_model(X, Y_and)
model_or  = train_model(X, Y_or)
model_xor = train_model(X, Y_xor)

# Predictions
with torch.no_grad():
    print("AND Gate Predictions:")
    print(model_and(X).round())  # Rounded to 0/1
    print("OR Gate Predictions:")
    print(model_or(X).round())
    print("XOR Gate Predictions:")
    print(model_xor(X).round())

def accuracy(model, X, Y):
    with torch.no_grad():
        preds = model(X).round()
        correct = (preds == Y).sum().item() # .item() extracts tensor(num) to python int(num)
        return correct / len(Y)

print("AND Accuracy:", accuracy(model_and, X, Y_and))
print("OR Accuracy:", accuracy(model_or, X, Y_or))
print("XOR Accuracy:", accuracy(model_xor, X, Y_xor))


AND Gate Predictions:
tensor([[0.],
        [0.],
        [0.],
        [1.]])
OR Gate Predictions:
tensor([[0.],
        [1.],
        [1.],
        [1.]])
XOR Gate Predictions:
tensor([[0.],
        [0.],
        [0.],
        [0.]])
AND Accuracy: 1.0
OR Accuracy: 1.0
XOR Accuracy: 0.5


# 2. Manual Perceptron Implementation

In [21]:
X = torch.tensor([[0.,0.],
                  [0.,1.],
                  [1.,0.],
                  [1.,1.]], dtype=torch.float32)
Y = torch.tensor([0.,0.,0.,1.], dtype=torch.float32)

# Initialize weights and bias (small random)
w = torch.randn(2, dtype=torch.float32) * 0.1
b = torch.randn(1, dtype=torch.float32) * 0.1
lr = 0.1

# Printing all shapes
print(X.size())
print(Y.size())
print(w.size())
print(b.size())

def step_activation(z):
    return 1.0 if z >= 0.0 else 0.0

print("\nManual perceptron training for AND gate")
epochs = 10
for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}")
    for i in range(len(X)):
        x_i = X[i] # print(x_i.size()) == torch.Size([2])
        y_i = Y[i].item() # Remember Y[i] indexing on tensor also returns a tensor with single scalar value; So we need to extract that value using .item() function
        z = torch.dot(w, x_i) + b
        y_pred = step_activation(z.item())
        error = y_i - y_pred
        # Update
        w = w + lr * error * x_i
        b = b + lr * error
        print(f"Input {x_i.tolist()} Target {int(y_i)} Pred {int(y_pred)} Updated w {w.tolist()} b {b.item():.4f}")

# Final predictions
print("\nFinal manual perceptron predictions:")
for i in range(len(X)):
    z = torch.dot(w, X[i]) + b
    print(f"Input {X[i].tolist()} -> {int(step_activation(z.item()))}")

torch.Size([4, 2])
torch.Size([4])
torch.Size([2])
torch.Size([1])

Manual perceptron training for AND gate

Epoch 1
Input [0.0, 0.0] Target 0 Pred 1 Updated w [0.010818742215633392, 0.0938381478190422] b -0.0600
Input [0.0, 1.0] Target 0 Pred 1 Updated w [0.010818742215633392, -0.006161853671073914] b -0.1600
Input [1.0, 0.0] Target 0 Pred 0 Updated w [0.010818742215633392, -0.006161853671073914] b -0.1600
Input [1.0, 1.0] Target 1 Pred 0 Updated w [0.11081874370574951, 0.0938381478190422] b -0.0600

Epoch 2
Input [0.0, 0.0] Target 0 Pred 0 Updated w [0.11081874370574951, 0.0938381478190422] b -0.0600
Input [0.0, 1.0] Target 0 Pred 1 Updated w [0.11081874370574951, -0.006161853671073914] b -0.1600
Input [1.0, 0.0] Target 0 Pred 0 Updated w [0.11081874370574951, -0.006161853671073914] b -0.1600
Input [1.0, 1.0] Target 1 Pred 0 Updated w [0.21081873774528503, 0.0938381478190422] b -0.0600

Epoch 3
Input [0.0, 0.0] Target 0 Pred 0 Updated w [0.21081873774528503, 0.0938381478190422] b -0.

# 3. Effect of Feature Scaling on Perceptron Training

In [22]:
# Synthetic dataset
X, y = make_classification(n_samples=200, n_features=2, 
                           n_informative=2, n_redundant=0, 
                           n_clusters_per_class=1, random_state=42)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1,1)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_test_t  = torch.tensor(y_test, dtype=torch.float32).reshape(-1,1)

def train_model(X, Y, epochs=50, lr=0.1):
    model = Perceptron()
    criterion = nn.BCELoss()
    optimizer = optim.Adam(params=model.parameters(), lr=lr)
    losses = []
    
    for epoch in range(epochs):
        outputs = model(X)
        loss = criterion(outputs, Y)
        losses.append(loss.item()) # Saving loss history by extracting int using item
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")
    
    return model, losses

print("Training WITHOUT scaling")
model_raw, losses_raw = train_model(X_train_t, y_train_t)

# Accuracy
with torch.no_grad():
    preds = model_raw(X_test_t).round()
    acc_raw = (preds == y_test_t).float().mean().item()
print("Final Accuracy (No Scaling):", acc_raw)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train_s = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_s  = torch.tensor(X_test_scaled, dtype=torch.float32)

print("Training WITH scaling")
model_scaled, losses_scaled = train_model(X_train_s, y_train_t)

# Accuracy
with torch.no_grad():
    preds = model_scaled(X_test_s).round()
    acc_scaled = (preds == y_test_t).float().mean().item()
print("Final Accuracy (With Scaling):", acc_scaled)


Training WITHOUT scaling
Epoch 1, Loss: 1.0032
Epoch 2, Loss: 0.9081
Epoch 3, Loss: 0.8237
Epoch 4, Loss: 0.7519
Epoch 5, Loss: 0.6936
Epoch 6, Loss: 0.6477
Epoch 7, Loss: 0.6109
Epoch 8, Loss: 0.5794
Epoch 9, Loss: 0.5506
Epoch 10, Loss: 0.5236
Epoch 11, Loss: 0.4986
Epoch 12, Loss: 0.4763
Epoch 13, Loss: 0.4569
Epoch 14, Loss: 0.4408
Epoch 15, Loss: 0.4278
Epoch 16, Loss: 0.4175
Epoch 17, Loss: 0.4095
Epoch 18, Loss: 0.4031
Epoch 19, Loss: 0.3981
Epoch 20, Loss: 0.3938
Epoch 21, Loss: 0.3902
Epoch 22, Loss: 0.3869
Epoch 23, Loss: 0.3839
Epoch 24, Loss: 0.3813
Epoch 25, Loss: 0.3789
Epoch 26, Loss: 0.3768
Epoch 27, Loss: 0.3751
Epoch 28, Loss: 0.3737
Epoch 29, Loss: 0.3727
Epoch 30, Loss: 0.3719
Epoch 31, Loss: 0.3714
Epoch 32, Loss: 0.3711
Epoch 33, Loss: 0.3709
Epoch 34, Loss: 0.3707
Epoch 35, Loss: 0.3706
Epoch 36, Loss: 0.3705
Epoch 37, Loss: 0.3704
Epoch 38, Loss: 0.3703
Epoch 39, Loss: 0.3702
Epoch 40, Loss: 0.3701
Epoch 41, Loss: 0.3701
Epoch 42, Loss: 0.3700
Epoch 43, Loss: 0.

# 4. Loss Function Comparison

In [23]:
# Dataset
X, y = make_classification(n_samples=200, n_features=2, 
                           n_informative=2, n_redundant=0, 
                           n_clusters_per_class=1, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_test_t  = torch.tensor(y_test.reshape(-1,1), dtype=torch.float32)

# Perceptron
class Perceptron(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(2,1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.fc(x))

def train_model(loss_fn, epochs=50, lr=0.1):
    model = Perceptron()
    criterion = loss_fn
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    losses = []
    
    for epoch in range(epochs):
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)
        losses.append(loss.item())
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    # Accuracy
    with torch.no_grad():
        preds = model(X_test_t).round()
        acc = (preds == y_test_t).float().mean().item()
    return model, losses, acc
# Binary Cross Entropy Loss
model_bce, losses_bce, acc_bce = train_model(nn.BCELoss())

# Mean Squared Error Loss
model_mse, losses_mse, acc_mse = train_model(nn.MSELoss())

print("Final Accuracy with BCELoss:", acc_bce)
print("Final Accuracy with MSELoss:", acc_mse)
print("Final Loss (BCELoss):", losses_bce[-1])
print("Final Loss (MSELoss):", losses_mse[-1])


Final Accuracy with BCELoss: 0.8500000238418579
Final Accuracy with MSELoss: 0.7749999761581421
Final Loss (BCELoss): 0.40372711420059204
Final Loss (MSELoss): 0.14019648730754852


# 5. Model Architecture and Activation Experiment

In [24]:
# Dataset
X, y = make_classification(n_samples=200, n_features=2, 
                           n_informative=2, n_redundant=0, 
                           n_clusters_per_class=1, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_test_t  = torch.tensor(y_test.reshape(-1,1), dtype=torch.float32)

# Sigmoid Perceptron
class SigmoidPerceptron(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(2,1)
        self.act = nn.Sigmoid()
    def forward(self, x):
        return self.act(self.fc(x))

# ReLU Perceptron
class ReLUPerceptron(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(2,1)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.act(self.fc(x))

def train_model(model_class, epochs=50, lr=0.1):
    model = model_class()
    criterion = nn.BCELoss() if isinstance(model, SigmoidPerceptron) else nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    with torch.no_grad():
        preds = model(X_test_t)
        if isinstance(model, SigmoidPerceptron):
            preds = preds.round()
        else:
            preds = (preds >= 0.5).float()  # thresholding ReLU outputs
        acc = (preds == y_test_t).float().mean().item()
        out_range = (outputs.min().item(), outputs.max().item())
    return acc, out_range

acc_sigmoid, range_sigmoid = train_model(SigmoidPerceptron)
acc_relu, range_relu = train_model(ReLUPerceptron)

print("Sigmoid Accuracy:", acc_sigmoid)
print("Sigmoid Output Range:", range_sigmoid)
print("ReLU Accuracy:", acc_relu)
print("ReLU Output Range:", range_relu)


Sigmoid Accuracy: 0.8500000238418579
Sigmoid Output Range: (0.03377271443605423, 0.9733083844184875)
ReLU Accuracy: 0.574999988079071
ReLU Output Range: (0.0, 0.24757128953933716)
